# Flagship experiment — does context change how we read two delivery records?

**All data in this notebook is synthetic.** It is generated with a fixed seed by `meritvector_analytics.synthetic`.

**Question.** Two employees deliver different projects. Employee A works on relatively simple, well-resourced projects
and is consistently on time. Employee B works on complex, constrained, dependency-heavy projects with more schedule
variability. What do raw metrics say, what does a context-adjusted model say, how uncertain is it, and how sensitive
is the answer to modelling assumptions?

**Ground rules.**
* A and B are chosen from *observable* data by a rule fixed in advance — never from hidden ground truth.
* No winner is predetermined. If the evidence is inadequate or assumptions fail, the notebook says so.
* Adjusted estimates are associations conditional on measured context, not causal effects of a person.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

from meritvector_analytics.synthetic import SyntheticConfig, generate
from meritvector_analytics.features import build_features
from meritvector_analytics.contextual import run_contextual
from meritvector_analytics.validation import sensitivity
from meritvector_analytics.experiment import run_experiment

SEED = 20261013
data = generate(SyntheticConfig(seed=SEED))
data.metadata["counts"]

{'employees': 300,
 'program_managers': 30,
 'project_managers': 200,
 'technical_project_managers': 70,
 'projects': 859,
 'milestones': 4599,
 'events': 68662,
 'meetings': 406,
 'business_units': 5}

## 1. Evidence → engineered features
Only validated `assessment_evidence` records are used. Milestones planned within 90 days of the as-of date are excluded
(right-censoring: recent late milestones are not yet observable, so the recent sample is optimistically biased).

In [2]:
fs = build_features(data.events, data.employees, data.config.as_of)
m = fs.milestones
print(f"{len(m)} achieved milestones, {m.employee_id.nunique()} employees, {m.project_id.nunique()} projects; "
      f"{fs.censored_excluded} recent milestones excluded for censoring")
m[["technical_complexity", "dependency_count", "resource_constraint", "scope_changes_to_date", "team_size",
   "schedule_variance_pct", "quality_score"]].describe().round(2)

3751 achieved milestones, 298 employees, 819 projects; 505 recent milestones excluded for censoring


,technical_complexity,dependency_count,resource_constraint,scope_changes_to_date,team_size,schedule_variance_pct,quality_score
count,3751.00,3751.00,3751.00,3751.00,3751.00,3751.00,3327.00
mean,3.10,2.58,0.32,0.68,7.88,10.10,3.14
std,1.11,2.00,0.19,0.93,3.01,15.63,0.67
min,1.00,0.00,0.01,0.00,3.00,-27.27,1.10
25%,2.00,1.00,0.17,0.00,6.00,0.00,2.70
50%,3.00,2.00,0.28,0.00,7.00,9.76,3.10
75%,4.00,4.00,0.44,1.00,10.00,20.69,3.60
max,5.00,11.00,0.89,6.00,21.00,80.00,5.00


## 2. How much do operating conditions matter? (context OLS, project-clustered SEs)

In [3]:
ctx = run_contextual(m, "schedule_variance_pct")
print(f"R² of context alone: {ctx.ols.rsquared:.3f}")
ctx.coefficients.round(3)

R² of context alone: 0.432


,term,estimate,ci_low,ci_high,p_value
0,Intercept,-16.611,-21.041,-12.180,0.000
1,C(environment)[T.internal],-1.933,-3.418,-0.449,0.011
2,C(environment)[T.regulated],3.144,1.422,4.866,0.000
3,C(maturity)[T.greenfield],4.960,3.612,6.308,0.000
4,C(job_family)[T.project_manager],0.107,-2.587,2.802,0.938
5,C(job_family)[T.technical_project_manager],2.551,-0.396,5.497,0.090
6,technical_complexity,2.638,1.855,3.422,0.000
7,dependency_count,2.260,1.858,2.663,0.000
8,resource_constraint,17.676,14.192,21.160,0.000
9,scope_changes_to_date,4.371,3.795,4.948,0.000


In [4]:
print("Variance decomposition (mixed model):", {k: round(v, 3) for k, v in ctx.variance_components.items()})
print("Assumption diagnostics:", {k: v for k, v in ctx.diagnostics.items() if k != "fit_warnings"})

Variance decomposition (mixed model): {'employee_intercept_var': 42.445, 'residual_var': 74.324, 'project_var': 25.072, 'employee_share_of_variance': 0.299}
Assumption diagnostics: {'n_observations': 3751, 'n_employees': 298, 'n_projects': 819, 'mixed_converged': True, 'project_variance_component': True, 'ols_r2': 0.4320066031204637, 'residual_skew': 0.2155931586513036, 'residual_excess_kurtosis': 3.642901726897742, 'breusch_pagan_p': 0.14771317902350783, 'assumption_notes': ['Residuals are heavy-tailed; intervals assume approximate normality and may be optimistic. Bootstrap and robust-regression sensitivity checks are reported alongside.', 'Assignment to projects is not random: adjusted estimates remove *measured* context only.']}


## 3. Select A and B (fixed rule, observable data only) and run the experiment

In [5]:
sens = sensitivity(m, ctx.mixed_estimates)
exp = run_experiment(m, ctx, sens["per_spec"], fs.employees)
print(exp["selection_rule"])
print("A =", exp["employees"]["A"], "| B =", exp["employees"]["B"])

Flagship experiment: does context change how we read two delivery records?

Employee A and Employee B are chosen *from observable data only* by a fixed,
documented rule — never from hidden ground truth and never to produce a chosen
winner:

* **A** — among people with >= 8 achieved milestones, raw on-time rate in the top
  quartile, choose the one whose work had the *lowest* context-expected slippage
  (simple projects, good resourcing).
* **B** — among people with >= 8 achieved milestones, raw on-time rate at or below
  the median, choose the one whose work had the *highest* context-expected
  slippage (complex, constrained, dependency-heavy projects).
A = SYN-E0201 | B = SYN-E0244


### 3.1 Raw performance metrics

In [6]:
pd.DataFrame(exp['raw_metrics']).round(3)

,A,B
milestones,21.000,13.000
projects,4.000,3.000
on_time_rate,0.952,0.000
mean_schedule_variance_pct,-15.277,32.153
schedule_variance_sd,10.096,16.129
quality_mean,3.360,3.054


### 3.2 Differences in operating conditions

In [7]:
pd.DataFrame(exp['operating_conditions']).round(2)

,A,B
technical_complexity,1.48,5.00
dependency_count,0.38,5.77
resource_constraint,0.06,0.49
scope_changes_to_date,0.00,1.54
team_size,8.81,8.77
size_band,2.33,2.00
regulated_share,0.00,0.85
greenfield_share,0.00,0.15
context_expected_variance_pct,-9.85,32.79


### 3.3 Context-adjusted estimates with 95% intervals
(pp of schedule variance; positive = less slippage)

In [8]:
pd.DataFrame(exp["adjusted_estimates"]).T

,adjusted_estimate,ci_low,ci_high,conclusion
A,3.423179,-2.140783,8.987141,within_expected_range
B,2.845167,-3.639199,9.329533,within_expected_range


In [9]:
print("Cluster-bootstrap B−A difference:", exp["difference_bootstrap"])

Cluster-bootstrap B−A difference: {'difference': -4.83218615492121, 'ci_low': -13.388837684153332, 'ci_high': 5.235686445679035, 'n_boot': 400}


### 3.4 Sensitivity to modelling assumptions

In [10]:
pd.DataFrame(exp['sensitivity']).round(2)

,specification,A,B,B_minus_A
0,base_mixed_model,3.42,2.85,-0.58
1,no_project_variance_component,2.16,3.39,1.23
2,add_business_unit_fixed_effects,3.09,2.35,-0.74
3,drop_resource_constraint,6.79,0.97,-5.82
4,drop_complexity_and_dependencies,6.78,-2.98,-9.75
5,winsorised_outcome_5_95,1.82,4.73,2.91
6,no_context_adjustment,14.51,-8.66,-23.17


In [11]:
print("Common support:", exp["common_support"])
print("Spec-level agreement with the base model across all employees:")
pd.DataFrame(sens["specifications"]).T.round(3)

Common support: {'by_feature': {'technical_complexity': False, 'dependency_count': False, 'resource_constraint': False, 'scope_changes_to_date': True, 'team_size': True, 'size_band': True}, 'adequate': False}
Spec-level agreement with the base model across all employees:


,spearman_with_base,share_conclusions_changed
no_project_variance_component,0.992,0.211
add_business_unit_fixed_effects,0.985,0.067
drop_resource_constraint,0.929,0.138
drop_complexity_and_dependencies,0.728,0.228
winsorised_outcome_5_95,0.993,0.023
no_context_adjustment,0.532,0.272
huber_robust_residuals,0.984,NaN


## 4. Verdict

In [12]:
print(exp["adjustment_summary"])
print()
print("STATUS:", exp["status"])
print(exp["verdict"])
print()
print(exp["why_raw_can_mislead"])

Without context adjustment the model puts Employee A ahead by 23.2 pp of schedule variance (raw on-time rates 95% vs 0%). After adjusting for operating conditions the B−A difference is -0.6 pp (bootstrap 95% CI -13.4 to +5.2).

STATUS: inconclusive
Inconclusive: the evidence does not support a confident comparison. Operating conditions barely overlap, so the adjustment extrapolates beyond observed data. The direction of the adjusted difference changes across reasonable model specifications. The bootstrap interval for the adjusted B−A difference includes zero.

Raw on-time rates mix the person's contribution with the difficulty of the work they were given. Employee B's projects carried higher complexity, tighter resourcing and more dependencies, each of which the context model associates with additional slippage. Comparing raw rates therefore penalises whoever is assigned harder work.


## 5. Synthetic ground-truth check (evaluation only)
Possible only because the data is synthetic. The hidden parameters are read *after* the analysis and were never
available to any model.

In [13]:
gt = data.ground_truth.set_index("employee_id")
mm = m[["employee_id", "month_index"]].join(gt[["true_contribution", "growth_per_year"]], on="employee_id")
mm["skill_t"] = mm.true_contribution + mm.growth_per_year * mm.month_index / 12
truth = mm.groupby("employee_id").skill_t.mean()
true_effect = 8.0 * (truth - truth.mean())  # generator coefficient: 8 pp per unit of latent contribution
a, b = exp["employees"]["A"], exp["employees"]["B"]
print(pd.DataFrame({"archetype (hidden)": gt.loc[[a, b], "archetype"].values,
                    "true effect (pp)": true_effect.loc[[a, b]].round(2).values,
                    "adjusted estimate": [exp["adjusted_estimates"][k]["adjusted_estimate"] for k in "AB"]}, index=["A", "B"]))
est = ctx.mixed_estimates.set_index("employee_id")
raw = ctx.raw.set_index("employee_id")
print("Spearman(truth, adjusted) =", round(truth.corr(est.adjusted_estimate, method="spearman"), 3))
print("Spearman(truth, raw)      =", round(truth.corr(raw.raw_estimate, method="spearman"), 3))

            archetype (hidden)  true effect (pp)  adjusted estimate
A  favourable_projects_average             -1.44           3.423179
B    difficult_projects_strong              6.20           2.845167
Spearman(truth, adjusted) = 0.774
Spearman(truth, raw)      = 0.284


**Reading the check.** If the verdict was *inconclusive* while the hidden truth shows a difference, that is the
system working as intended: the evidence for these two people (few milestones, little overlap in conditions) was not
strong enough to support a confident individual comparison, even though adjustment moved the estimates in the right
direction across the population.

## 6. Reproducibility

In [14]:
again = run_experiment(m, ctx, sens["per_spec"], fs.employees)
assert json.dumps(again, sort_keys=True, default=str) == json.dumps(exp, sort_keys=True, default=str)
data2 = generate(SyntheticConfig(seed=SEED))
assert data2.events.equals(data.events)
print("Reproducible: identical data and identical experiment output for seed", SEED)

Reproducible: identical data and identical experiment output for seed 20261013
